In [27]:

import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
import ast
import pickle

In [28]:
df1 = pd.read_csv("../data/expenses_2023_2024_2025.csv")
df2 = pd.read_csv("../data/expenses_2026.csv")
len(df1) , len(df2)

(724, 65)

In [29]:
df = pd.concat([df1, df2], ignore_index = True)
df

,Date,Name,Category,Amount,Notes,Receipt,Created
0,2026/02/04,pte,Education (https://www.notion.so/Education-65f...,CA$406.00,NaN,NaN,"February 4, 2026 7:50 AM"
1,2026/02/04,aws,Utilities (https://www.notion.so/Utilities-9c7...,CA$3.54,NaN,NaN,"February 4, 2026 7:49 AM"
2,2026/02/04,lyft,Transport (https://www.notion.so/Transport-8c5...,CA$9.84,NaN,NaN,"February 4, 2026 7:49 AM"
3,2026/02/04,mima cake smol,Food (https://www.notion.so/Food-ceb4c1b542e64...,CA$12.50,NaN,NaN,"February 4, 2026 7:49 AM"
4,2026/02/04,lyft,Transport (https://www.notion.so/Transport-8c5...,CA$7.14,NaN,NaN,"February 4, 2026 7:48 AM"
...,...,...,...,...,...,...,...
784,2026/01/01,popeyes,Food (https://www.notion.so/Food-2fdf202d177d8...,CA$21.76,NaN,NaN,"February 4, 2026 2:57 AM"
785,2026/01/01,rent,Rent (https://www.notion.so/Rent-2fdf202d177d8...,"CA$1,000.00",NaN,NaN,"February 4, 2026 2:57 AM"
786,2026/01/01,uber eats,Food (https://www.notion.so/Food-2fdf202d177d8...,CA$20.84,NaN,NaN,"February 4, 2026 2:57 AM"
787,2026/01/01,walmart,Groceries (https://www.notion.so/Groceries-2fd...,CA$61.14,NaN,NaN,"February 4, 2026 2:57 AM"


In [30]:
df.drop(columns = ["Date","Amount","Receipt","Created","Notes"], inplace = True)
df

,Name,Category
0,pte,Education (https://www.notion.so/Education-65f...
1,aws,Utilities (https://www.notion.so/Utilities-9c7...
2,lyft,Transport (https://www.notion.so/Transport-8c5...
3,mima cake smol,Food (https://www.notion.so/Food-ceb4c1b542e64...
4,lyft,Transport (https://www.notion.so/Transport-8c5...
...,...,...
784,popeyes,Food (https://www.notion.so/Food-2fdf202d177d8...
785,rent,Rent (https://www.notion.so/Rent-2fdf202d177d8...
786,uber eats,Food (https://www.notion.so/Food-2fdf202d177d8...
787,walmart,Groceries (https://www.notion.so/Groceries-2fd...


In [31]:
df[df.isna().any(axis=1)]


,Name,Category
55,New Expense,NaN
138,New Expense,NaN
311,New Expense,NaN
788,New Expense,NaN


In [32]:
df.dropna(inplace = True)
df[df.isna().any(axis=1)]

,Name,Category


In [33]:
df["Category_clean"] = df["Category"].str.replace(r" .*",'',regex = True)

df["Category_clean"]

0      Education
1      Utilities
2      Transport
3           Food
4      Transport
         ...    
783         Food
784         Food
785         Rent
786         Food
787    Groceries
Name: Category_clean, Length: 785, dtype: str

In [34]:
df.drop(columns = "Category",inplace = True)

In [35]:
model = SentenceTransformer("all-MiniLM-L6-v2")
df["embeddings"] = model.encode(df["Name"].tolist()).tolist()
df.to_csv("../data/expenses_embeddings.csv")

with open("embedding_model.pkl", 'wb') as file:
    pickle.dump(model,file)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9104.41it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
print(df["embeddings"].iloc[0])
print(type(df["embeddings"].iloc[0]))

In [ ]:
stored_embeddings = np.array(df["embeddings"].tolist())
stored_embeddings

In [ ]:
def cosine_similarity(a, b):
    return np.dot(a, b.T) / (np.linalg.norm(a) * np.linalg.norm(b, axis=1))

In [ ]:
def categorize(merchant_name: str, threshold: float = 0.90):
    query_embedding = model.encode([merchant_name])[0]
    #print("query = ",query_embedding)
    scores = cosine_similarity(query_embedding, stored_embeddings)
    #print("score = ",scores)
    
    top_indices = scores.argsort()[::-1][:3]  # top 3 matches
    
    top_score = scores[top_indices[0]]
    top_category = df.iloc[top_indices[0]]["Category_clean"]
    top_merchant = df.iloc[top_indices[0]]["Name"]

    print(top_indices)
    print(top_score)
    print(top_category)
    print(top_merchant)
    #return
    if top_score >= threshold:
        return {
            "category": top_category,
            "confidence": float(top_score),
            "matched_to": top_merchant,
            "source": "embeddings"
        }
    else:
        top_matches = [
            (df.iloc[i]["Name"], df.iloc[i]["Category_clean"], float(scores[i]))
            for i in top_indices
        ]
        return {
            "category": None,
            "confidence": float(top_score),
            "top_matches": top_matches,
            "source": "needs_llm"
        }

In [ ]:
result = categorize("sharwma")
result